In [1]:
# Cell 1 — Mount Drive and extract DICOMs
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import tarfile

DATA_DIR  = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS'
os.makedirs(DICOM_DIR, exist_ok=True)

print('Extracting DICOMs...')
with tarfile.open(f'{DATA_DIR}/fastmri_prostate_DICOMS_IDS_001_312.tar', 'r') as tar:
    tar.extractall(DICOM_DIR)

DICOM_DIR = '/content/DICOMS/DICOMS'
print(f'Patients: {len(os.listdir(DICOM_DIR))}')

Mounted at /content/drive
Extracting DICOMs...


/tmp/ipykernel_6556/2693115195.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DICOM_DIR)


Patients: 312


In [2]:
# Cell 2 — Installs
!pip install pydicom h5py optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.2 MB/s eta 0:00:00


In [3]:
# Cell 3 — Imports
import os
import numpy as np
import pandas as pd
import tarfile
import pydicom
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from PIL import Image
import torchvision.transforms as transforms
import optuna
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Cell 4 — Load labels and create splits
DATA_DIR  = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS/DICOMS'

labels_tar = f'{DATA_DIR}/labels.tar'
with tarfile.open(labels_tar, 'r') as tar:
    f = tar.extractfile('labels/t2_slice_level_labels.csv')
    t2_labels = pd.read_csv(f)
    f = tar.extractfile('labels/volume_exam_labels.csv')
    volume_labels = pd.read_csv(f)

volume_labels['binary_label'] = (volume_labels['t2_volume_level'] >= 3).astype(int)
patient_split = t2_labels.drop_duplicates('fastmri_pt_id')[['fastmri_pt_id', 'data_split']]
df = patient_split.merge(
    volume_labels[['fastmri_pt_id', 'binary_label', 't2_volume_level']],
    on='fastmri_pt_id'
)

# Remove invalid patients with empty T2 folders
invalid_ids = [115, 258]
train_df = df[df['data_split'] == 'training'].reset_index(drop=True)
val_df   = df[df['data_split'] == 'validation'].reset_index(drop=True)
test_df  = df[df['data_split'] == 'test'].reset_index(drop=True)

train_df = train_df[~train_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
val_df   = val_df[~val_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
test_df  = test_df[~test_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(df['binary_label'].value_counts())

Train: 217, Val: 47, Test: 46
binary_label
0    178
1    134
Name: count, dtype: int64


In [5]:
# Cell 5 — Dataset class (3x3 grid of middle 9 slices)
class ProstateT2Dataset(Dataset):
    def __init__(self, df, dicom_dir, augment=False):
        self.df = df.reset_index(drop=True)
        self.dicom_dir = dicom_dir

        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])

        print(f'Dataset: {len(self.df)} patients')
        print(f'Labels: {self.df.binary_label.value_counts().to_dict()}')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pt_id = str(row.fastmri_pt_id).zfill(3)
        t2_path = os.path.join(self.dicom_dir, pt_id, 'AX_T2')

        slices = []
        for fname in os.listdir(t2_path):
            if fname.endswith('.dcm'):
                ds = pydicom.dcmread(os.path.join(t2_path, fname))
                slices.append((int(ds.InstanceNumber), ds.pixel_array.astype(np.float32)))

        slices.sort(key=lambda x: x[0])
        volume = np.stack([s[1] for s in slices])  # (N, 320, 320)

        # Per-volume min-max normalization
        volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

        # Take middle 9 slices and tile into 3x3 grid
        mid = volume.shape[0] // 2
        selected = volume[mid-4:mid+5]  # 9 slices

        rows = []
        for i in range(3):
            row_img = np.concatenate([selected[i*3], selected[i*3+1], selected[i*3+2]], axis=1)
            rows.append(row_img)
        grid = np.concatenate(rows, axis=0)  # (960, 960)

        img = Image.fromarray((grid * 255).astype(np.uint8))
        img = img.convert('RGB')
        img = self.transform(img)

        label = torch.tensor(row.binary_label, dtype=torch.float32)
        return img, label

In [6]:
# Cell 6 — DataLoaders
train_dataset = ProstateT2Dataset(train_df, DICOM_DIR, augment=True)
val_dataset   = ProstateT2Dataset(val_df,   DICOM_DIR, augment=False)
test_dataset  = ProstateT2Dataset(test_df,  DICOM_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=0)

# Test one sample
sample_img, sample_label = train_dataset[0]
print(f'Image shape: {sample_img.shape}')
print(f'Label: {sample_label}')

Dataset: 217 patients
Labels: {0: 129, 1: 88}
Dataset: 47 patients
Labels: {1: 26, 0: 21}
Dataset: 46 patients
Labels: {0: 26, 1: 20}
Image shape: torch.Size([3, 224, 224])
Label: 0.0


In [7]:
# Cell 7 — EfficientNet-B3 model definition and device
def get_model(dropout=0.5):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    # Freeze early layers (first 4 blocks)
    for i, block in enumerate(model.features):
        if i < 4:
            for param in block.parameters():
                param.requires_grad = False
    # Replace classifier head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 1)
    )
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Test forward pass
test_model  = get_model().to(device)
test_input  = torch.randn(2, 3, 224, 224).to(device)
test_output = test_model(test_input)
print(f'Output shape: {test_output.shape}')  # should be (2, 1)
total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M')
print(f'Trainable params: {trainable_params/1e6:.1f}M')
del test_model, test_input, test_output

Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 231MB/s]


Output shape: torch.Size([2, 1])
Total params: 10.7M
Trainable params: 10.5M


In [8]:
# Cell 8 — Train/Val functions
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc

In [9]:
# Cell 9 — Optuna hyperparameter search
n_neg = (train_df['binary_label'] == 0).sum()
n_pos = (train_df['binary_label'] == 1).sum()

def objective(trial):
    lr_backbone  = trial.suggest_float('lr_backbone',  1e-7, 1e-4, log=True)
    lr_head      = trial.suggest_float('lr_head',      1e-5, 1e-3, log=True)
    dropout      = trial.suggest_float('dropout',      0.1,  0.6)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 0.1,  log=True)

    trial_model = get_model(dropout=dropout).to(device)

    backbone_params   = [p for p in trial_model.parameters()
                         if p.requires_grad and id(p) not in
                         set(id(p) for p in trial_model.classifier.parameters())]
    classifier_params = list(trial_model.classifier.parameters())

    optimizer = torch.optim.AdamW([
        {'params': backbone_params,   'lr': lr_backbone},
        {'params': classifier_params, 'lr': lr_head}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

    pw        = torch.tensor([n_neg / n_pos]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    best_val_auc     = 0
    patience         = 5
    patience_counter = 0

    for epoch in range(20):
        train_epoch(trial_model, train_loader, optimizer, criterion, device)
        scheduler.step()
        _, val_auc = val_epoch(trial_model, val_loader, criterion, device)

        if val_auc > best_val_auc:
            best_val_auc     = val_auc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_auc


study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
)
study.optimize(objective, n_trials=20, timeout=7200)

print('\nBest EfficientNet trial:')
print(f'  Val AUC: {study.best_value:.4f}')
print(f'  Params: {study.best_params}')

[I 2026-05-18 02:16:13,661] A new study created in memory with name: no-name-6a92247b-76ff-4baa-b96e-510be7f79d35
[I 2026-05-18 02:26:51,321] Trial 0 finished with value: 0.63003663003663 and parameters: {'lr_backbone': 1.0957725677472038e-06, 'lr_head': 2.4224827205714012e-05, 'dropout': 0.3786540605490868, 'weight_decay': 0.035868732993498055}. Best is trial 0 with value: 0.63003663003663.
[I 2026-05-18 02:32:39,727] Trial 1 finished with value: 0.6282051282051282 and parameters: {'lr_backbone': 2.138235313448559e-05, 'lr_head': 6.356460107284168e-05, 'dropout': 0.44438671229991045, 'weight_decay': 0.0014364531973365584}. Best is trial 0 with value: 0.63003663003663.
[I 2026-05-18 02:44:52,940] Trial 2 finished with value: 0.554945054945055 and parameters: {'lr_backbone': 2.1211467071619506e-06, 'lr_head': 0.00012427668864147502, 'dropout': 0.29032896256478125, 'weight_decay': 0.016517491502811003}. Best is trial 0 with value: 0.63003663003663.
[I 2026-05-18 02:50:42,903] Trial 3 fin


Best EfficientNet trial:
  Val AUC: 0.7271
  Params: {'lr_backbone': 8.3978723660081e-06, 'lr_head': 0.0009227039627306356, 'dropout': 0.4919215370853721, 'weight_decay': 0.09934496726187109}


In [10]:
# Cell 10 — Initialize final model with best Optuna params
best = study.best_params
print(f'Training final EfficientNet model with: {best}')

model = get_model(dropout=best['dropout']).to(device)

backbone_params   = [p for p in model.parameters()
                     if p.requires_grad and id(p) not in
                     set(id(p) for p in model.classifier.parameters())]
classifier_params = list(model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {'params': backbone_params,   'lr': best['lr_backbone']},
    {'params': classifier_params, 'lr': best['lr_head']}
], weight_decay=best['weight_decay'])

scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
pos_weight = torch.tensor([n_neg / n_pos]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print('Ready')

Training final EfficientNet model with: {'lr_backbone': 8.3978723660081e-06, 'lr_head': 0.0009227039627306356, 'dropout': 0.4919215370853721, 'weight_decay': 0.09934496726187109}
Ready


In [11]:
# Cell 11 — Training loop
EPOCHS   = 50
PATIENCE = 10
best_val_auc     = 0
patience_counter = 0
best_model_path  = f'{DATA_DIR}/best_efficientnet.pth'

for epoch in range(EPOCHS):
    train_loss, train_auc = train_epoch(model, train_loader, optimizer, criterion, device)
    scheduler.step()
    val_loss, val_auc     = val_epoch(model, val_loader, criterion, device)

    if val_auc > best_val_auc:
        best_val_auc     = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'Epoch {epoch+1:02d}/{EPOCHS} — Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} *** BEST ***')
    else:
        patience_counter += 1
        print(f'Epoch {epoch+1:02d}/{EPOCHS} — Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} (patience {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}.')
            break

print(f'\nBest Val AUC: {best_val_auc:.4f}')

Epoch 01/50 — Train Loss: 0.8260 AUC: 0.5320 | Val Loss: 0.8530 AUC: 0.6227 *** BEST ***
Epoch 02/50 — Train Loss: 0.8142 AUC: 0.5802 | Val Loss: 0.8675 AUC: 0.6557 *** BEST ***
Epoch 03/50 — Train Loss: 0.8103 AUC: 0.5950 | Val Loss: 0.8741 AUC: 0.6282 (patience 1/10)
Epoch 04/50 — Train Loss: 0.8174 AUC: 0.5512 | Val Loss: 0.8841 AUC: 0.5934 (patience 2/10)
Epoch 05/50 — Train Loss: 0.8077 AUC: 0.6047 | Val Loss: 0.8896 AUC: 0.5293 (patience 3/10)
Epoch 06/50 — Train Loss: 0.8102 AUC: 0.5845 | Val Loss: 0.8951 AUC: 0.5476 (patience 4/10)
Epoch 07/50 — Train Loss: 0.7937 AUC: 0.6511 | Val Loss: 0.9162 AUC: 0.5073 (patience 5/10)
Epoch 08/50 — Train Loss: 0.8165 AUC: 0.5675 | Val Loss: 0.9226 AUC: 0.5128 (patience 6/10)
Epoch 09/50 — Train Loss: 0.7940 AUC: 0.6403 | Val Loss: 0.9136 AUC: 0.5330 (patience 7/10)
Epoch 10/50 — Train Loss: 0.7837 AUC: 0.6659 | Val Loss: 0.8950 AUC: 0.5678 (patience 8/10)
Epoch 11/50 — Train Loss: 0.7717 AUC: 0.6886 | Val Loss: 0.8594 AUC: 0.6813 *** BEST *

In [12]:
# Cell 12 — Test evaluation
model.load_state_dict(torch.load(best_model_path))
_, test_auc = val_epoch(model, test_loader, criterion, device)
print(f'EfficientNet Test AUC: {test_auc:.4f}')

EfficientNet Test AUC: 0.5500


In [13]:
# Cell 13 — Save results
import json

efficientnet_results = {
    'model': 'EfficientNet-B3',
    'approach': '3x3 grid of middle 9 slices',
    'best_val_auc': float(best_val_auc),
    'test_auc': float(test_auc),
    'best_params': study.best_params,
    'n_train': len(train_df),
    'n_val': len(val_df),
    'n_test': len(test_df),
    'epochs_trained': epoch + 1,
    'total_params_M': 12,
}

results_path = f'{DATA_DIR}/efficientnet_results.json'
with open(results_path, 'w') as f:
    json.dump(efficientnet_results, f, indent=2)

print('EfficientNet results saved:')
print(json.dumps(efficientnet_results, indent=2))

EfficientNet results saved:
{
  "model": "EfficientNet-B3",
  "approach": "3x3 grid of middle 9 slices",
  "best_val_auc": 0.6813186813186813,
  "test_auc": 0.55,
  "best_params": {
    "lr_backbone": 8.3978723660081e-06,
    "lr_head": 0.0009227039627306356,
    "dropout": 0.4919215370853721,
    "weight_decay": 0.09934496726187109
  },
  "n_train": 217,
  "n_val": 47,
  "n_test": 46,
  "epochs_trained": 21,
  "total_params_M": 12
}
